In [ ]:
# ── Google Colab setup — mount Drive and set project root ─────────────────
import os, sys

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    # Adjust this path if your folder is nested differently in Drive
    PROJECT_ROOT_OVERRIDE = '/content/drive/MyDrive/Language-Project-main'
    assert os.path.isdir(PROJECT_ROOT_OVERRIDE), \
        f'Folder not found: {PROJECT_ROOT_OVERRIDE}\nCheck your Drive path.'
    print(f'Drive mounted. Project root: {PROJECT_ROOT_OVERRIDE}')
    os.chdir(PROJECT_ROOT_OVERRIDE)
except ModuleNotFoundError:
    # Not running on Colab — local run, leave path detection to find_project_root()
    PROJECT_ROOT_OVERRIDE = None
    print('Local run detected — Drive mount skipped.')

# GemmaX2-28-2B Full-Transcript Embeddings (Dual-Model)

Generates **GemmaX2-28-2B** contextual embeddings for all three languages (EN, HE, AR)
using the complete podcast transcript as context.

## Dual-model design

Two models are used, each for the task they are best suited for:

| Model          | Role                          | Why                                      |
|----------------|-------------------------------|------------------------------------------|
| XLM-RoBERTa    | Word location (HE/AR only)    | Trained for cross-lingual semantic       |
|                | via cosine similarity         | similarity — reliable matching signal    |
| GemmaX2-28-2B      | Final embedding extraction    | Autoregressive, left-to-right, matches   |
|                | from full-context prefix      | real-time speech processing in the brain |

For Hebrew and Arabic, XLM-RoBERTa locates the target word's position within
its sentence using cosine similarity (same approach as notebooks 02/03).
Once the position is confirmed, the final embedding is taken from GemmaX2's
hidden state at that position in the full-context prefix.

This separates two concerns cleanly:
- **Word location**: XLM-RoBERTa's bidirectional representations are geometrically
  comparable across contexts, making cosine similarity a reliable matching criterion.
- **Representation quality**: GemmaX2's autoregressive hidden states reflect real-time
  causal processing, giving better neural encoding performance.

For English, XLM-RoBERTa is not needed — the target word is located directly
in the full transcript by timestamp matching.

## Context sources

| Language | Context fed to GemmaX2                    |
|----------|----------------------------------------|
| English  | `podcast_transcript.csv` (5,136 words) |
| Hebrew   | 402 translated sentences concatenated  |
| Arabic   | 402 translated sentences concatenated  |

## Memory requirements

Both models loaded simultaneously:
- XLM-RoBERTa-base : ~500MB
- GemmaX2-28-2B (fp16) : ~3.5GB
- Total             : ~4GB VRAM — fits comfortably on an 8GB GPU

XLM-RoBERTa is used only for HE/AR cosine matching, then its outputs are
discarded. GemmaX2 provides all final embeddings.

## Input files
- `podcast_transcript.csv` — 5,136-word English transcript with timestamps
- `translated_podcast_transcript_filtered.csv` — 1,735 content words
  with en/he/ar columns and timestamps
- `podcast_sentences_en/he/ar.csv` — 402 parallel sentences per language

## Output files
- `en_gemmax2_embeddings.csv` — (1735, 2304)
- `he_gemmax2_embeddings.csv` — (1735, 2304)
- `ar_gemmax2_embeddings.csv` — (1735, 2304)
- `gemmax2_embedding_report.csv` — per-word context length, similarity, match flags

## 1. Imports

In [ ]:
import re
import csv
import pandas as pd
import torch
import numpy as np
import unicodedata
from pathlib import Path
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM
from tqdm import tqdm

def find_project_root(start=None):
    # Honour Colab Drive override if set
    override = globals().get('PROJECT_ROOT_OVERRIDE')
    if override and Path(override).is_dir():
        return Path(override)
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in [start, *start.parents]:
        if (path / 'data').exists() and (path / 'notebooks').exists():
            return path
    raise FileNotFoundError(
        'Could not find project root.\n'
        'On Colab: run the first cell to mount Drive before this one.'
    )

PROJECT_ROOT  = find_project_root()
DATA_DIR      = PROJECT_ROOT / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'
SENTENCES_DIR = DATA_DIR / 'sentences'

print(f'Project root : {PROJECT_ROOT}')

## 2. Configuration

In [ ]:
# ── Models ────────────────────────────────────────────────────────────────────
GemmaX2_MODEL_NAME  = 'ModelSpace/GemmaX2-28-2B-v0.1'
XLM_MODEL_NAME   = 'xlm-roberta-base'
HIDDEN_DIM       = 2304   # GemmaX2-28-2B hidden dimension
XLM_HIDDEN_DIM   = 768    # XLM-RoBERTa-base hidden dimension

# ── Context window ─────────────────────────────────────────────────────────────
MAX_TOKENS               = 2048
TARGET_TOKEN_RESERVE     = 10
EFFECTIVE_CONTEXT_TOKENS = MAX_TOKENS - TARGET_TOKEN_RESERVE

# ── Word location threshold (XLM-RoBERTa cosine similarity) ───────────────────
# Same threshold used in notebooks 02/03 — validated for XLM-RoBERTa.
# Words below this are dropped (zero vector) and flagged in the report.
SIM_THRESHOLD = 0.70

LANGUAGES = ['en', 'he', 'ar']

OUTPUT_PATHS = {
    'en': PROCESSED_DIR / 'en_gemmax2_embeddings.csv',
    'he': PROCESSED_DIR / 'he_gemmax2_embeddings.csv',
    'ar': PROCESSED_DIR / 'ar_gemmax2_embeddings.csv',
}
REPORT_PATH = PROCESSED_DIR / 'gemmax2_embedding_report.csv'

print(f'GemmaX2 model         : {GemmaX2_MODEL_NAME}')
print(f'XLM-RoBERTa model  : {XLM_MODEL_NAME}  (word location only)')
print(f'GemmaX2 hidden dim    : {HIDDEN_DIM}')
print(f'Max context tokens : {MAX_TOKENS}')
print(f'Sim threshold      : {SIM_THRESHOLD}')

## 3. Load Data

In [ ]:
# ── Filtered content words ────────────────────────────────────────────────────
FILTERED_CANDIDATES = [
    DATA_DIR / 'Amirim_Project_Submission' / 'translated_podcast_transcript_filtered.csv',
    DATA_DIR / 'Amirim_Project_Submission' / 'Amirim_Project_Submission' / 'translated_podcast_transcript_filtered.csv',
]
filtered_path = next((p for p in FILTERED_CANDIDATES if p.exists()), None)
if filtered_path is None:
    raise FileNotFoundError('translated_podcast_transcript_filtered.csv not found')
content_df = pd.read_csv(filtered_path)
print(f'Content words   : {len(content_df)}')

# ── Full English transcript ───────────────────────────────────────────────────
full_df = pd.read_csv(DATA_DIR / 'ds005574' / 'stimuli' / 'podcast_transcript.csv')
print(f'Full transcript : {len(full_df)} words')

# ── 402 parallel sentences ────────────────────────────────────────────────────
def load_sentences_csv(path):
    sentences = []
    with open(path, encoding='utf-8-sig') as f:
        reader = csv.reader(f)
        next(reader, None)
        for row in reader:
            if not row or row[0] == 'sentence_id':
                continue
            sentences.append(row[2] if len(row) >= 3 else row[1])
    return sentences

en_sentences = load_sentences_csv(SENTENCES_DIR / 'podcast_sentences_en.csv')
he_sentences = load_sentences_csv(SENTENCES_DIR / 'podcast_sentences_he.csv')
ar_sentences = load_sentences_csv(SENTENCES_DIR / 'podcast_sentences_ar.csv')

assert len(en_sentences) == len(he_sentences) == len(ar_sentences)
print(f'Sentences       : {len(en_sentences)}')

# ── Full HE/AR texts (concatenated sentences) ─────────────────────────────────
# Joining with a space gives GemmaX2 one continuous text stream —
# the same structure as the English full transcript.
he_full_text = ' '.join(he_sentences)
ar_full_text = ' '.join(ar_sentences)
en_full_text = ' '.join(str(w) for w in full_df['word'].tolist())

# Pre-compute char end positions for every word in the English full transcript
# so we can quickly slice the prefix up to any target word
en_word_char_ends = []
pos = 0
for w in full_df['word'].tolist():
    end = pos + len(str(w))
    en_word_char_ends.append(end)
    pos = end + 1  # +1 for joining space

# Pre-compute char start positions for each sentence in HE/AR concatenated text
def sent_char_starts(sentences):
    starts, pos = [], 0
    for s in sentences:
        starts.append(pos)
        pos += len(s) + 1
    return starts

he_sent_char_starts = sent_char_starts(he_sentences)
ar_sent_char_starts = sent_char_starts(ar_sentences)

print('Text streams ready.')
print(f'  EN : {len(en_full_text.split())} words')
print(f'  HE : {len(he_full_text.split())} words (approx)')
print(f'  AR : {len(ar_full_text.split())} words (approx)')

## 4. Build Sentence Time Boundaries

Assigns each HE/AR content word to its sentence by timestamp.
Identical to the approach in notebooks 02/03.

In [ ]:
def norm_en(text):
    return re.sub(r'[^a-z0-9]', '', text.lower())


full_words_norm = [norm_en(w) for w in full_df['word'].tolist()]
full_starts_arr = full_df['start'].values
n_full          = len(full_words_norm)

sentence_boundaries = {}
ptr = 0
for sent_idx, sentence in enumerate(en_sentences):
    sent_words = [norm_en(w) for w in sentence.split() if norm_en(w)]
    if not sent_words:
        continue
    for offset in range(min(200, n_full - ptr)):
        start = ptr + offset
        if start >= n_full:
            break
        if full_words_norm[start] == sent_words[0]:
            if len(sent_words) < 2 or (
                start + 1 < n_full and
                full_words_norm[start + 1] == sent_words[1]
            ):
                t_start = full_starts_arr[start]
                t_end   = full_starts_arr[min(start + len(sent_words) - 1, n_full - 1)]
                sentence_boundaries[sent_idx] = (t_start, t_end)
                ptr = start + len(sent_words)
                break
    else:
        if sent_idx > 0 and sent_idx - 1 in sentence_boundaries:
            prev_end = sentence_boundaries[sent_idx - 1][1]
            sentence_boundaries[sent_idx] = (prev_end, prev_end)

boundary_list = sorted(
    [(v[0], v[1], k) for k, v in sentence_boundaries.items()]
)
print(f'Sentences with boundaries : {len(sentence_boundaries)} / {len(en_sentences)}')


def find_sentence_for_time(t):
    best_si, best_dist = None, float('inf')
    for (s, e, si) in boundary_list:
        if s <= t <= e:
            return si
        dist = min(abs(t - s), abs(t - e))
        if dist < best_dist:
            best_dist, best_si = dist, si
    return best_si


word_to_sent = [
    find_sentence_for_time(content_df.iloc[wi]['start'])
    for wi in range(len(content_df))
]
print(f'Content words assigned : {sum(1 for s in word_to_sent if s is not None)} / {len(content_df)}')


def find_full_transcript_idx(t):
    """Return index in full_df of the word whose timestamp is closest to t."""
    return int(np.argmin(np.abs(full_starts_arr - t)))

## 5. Load Both Models

XLM-RoBERTa is used only for Hebrew/Arabic word location via cosine similarity.
GemmaX2-28-2B produces all final embeddings.

In [ ]:
device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device : {device}')
print()

# ── XLM-RoBERTa (word location — HE/AR only) ──────────────────────────────────
print(f'Loading {XLM_MODEL_NAME} for word location...')
xlm_tokenizer = AutoTokenizer.from_pretrained(XLM_MODEL_NAME)
xlm_model     = AutoModel.from_pretrained(XLM_MODEL_NAME)
xlm_model     = xlm_model.to(device)
xlm_model.eval()
print(f'  XLM-RoBERTa ready  hidden_dim={xlm_model.config.hidden_size}')
print()

# ── GemmaX2-28-2B (final embeddings) ──────────────────────────────────────────────
print(f'Loading {GemmaX2_MODEL_NAME} for embedding extraction...')
print('  (~5GB in float16 — GPU recommended)')
gemmax2_tokenizer = AutoTokenizer.from_pretrained(GemmaX2_MODEL_NAME)
gemmax2_model     = AutoModelForCausalLM.from_pretrained(
    GemmaX2_MODEL_NAME,
    torch_dtype=torch.float16 if device == 'cuda' else torch.float32
)
gemmax2_model = gemmax2_model.to(device)
gemmax2_model.eval()
print(f'  GemmaX2-28-2B ready    hidden_dim={gemmax2_model.config.hidden_size}')

assert gemmax2_model.config.hidden_size == HIDDEN_DIM, \
    f'GemmaX2 hidden dim mismatch: {gemmax2_model.config.hidden_size} != {HIDDEN_DIM}'
print()
print('Both models loaded ✓')

## 6. Helper Functions

In [ ]:
def normalize_text(text, lang):
    """Diacritic stripping + punctuation removal for token matching only."""
    text = unicodedata.normalize('NFC', str(text))
    if lang == 'he':
        text = ''.join(c for c in text if not ('\u05B0' <= c <= '\u05C7'))
        text = text.replace('\u05F3', '').replace('\u05F4', '')
    elif lang == 'ar':
        text = ''.join(c for c in text if not ('\u064B' <= c <= '\u065F'))
        text = text.replace('\u0640', '')
    text = text.replace('\u2019', '').replace("'", '').replace('`', '')
    text = text.strip(' .,!?"()-:;[]{}\u060C\u061F\u061B\u05BE')
    return text.lower()


def cosine_sim(a, b):
    a = a / (np.linalg.norm(a) + 1e-8)
    b = b / (np.linalg.norm(b) + 1e-8)
    return float(np.dot(a, b))


# ── XLM-RoBERTa helpers (word location only) ──────────────────────────────────

xlm_sent_cache = {}   # sentence_text -> list of per-word averaged vectors

def get_xlm_word_vectors(sentence_text):
    """
    Run XLM-RoBERTa on sentence_text and return a list of (norm_text, vector)
    tuples — one per whitespace-split word, averaged over subword tokens.
    Results are cached by sentence text.
    """
    if sentence_text in xlm_sent_cache:
        return xlm_sent_cache[sentence_text]

    enc = xlm_tokenizer(
        sentence_text,
        return_tensors='pt',
        truncation=True,
        max_length=512
    ).to(device)
    with torch.no_grad():
        out = xlm_model(**enc)
    hidden = out.last_hidden_state.squeeze(0).cpu().numpy()
    word_ids = enc.word_ids() if hasattr(enc, 'word_ids') else \
               xlm_tokenizer(sentence_text, return_tensors='pt').word_ids()

    word_vecs = {}
    for tok_idx, wid in enumerate(word_ids):
        if wid is None:
            continue
        word_vecs.setdefault(wid, []).append(hidden[tok_idx])

    words     = sentence_text.split()
    result    = []
    for wid in sorted(word_vecs):
        avg  = np.mean(word_vecs[wid], axis=0).astype(np.float32)
        text = normalize_text(words[wid], 'he') if wid < len(words) else ''
        if text:
            result.append((text, avg))

    xlm_sent_cache[sentence_text] = result
    return result


def get_xlm_isolated_vec(target_text, lang):
    """
    Embed target_text in isolation using XLM-RoBERTa.
    Returns a single 768-d vector (mean over all tokens).
    """
    enc = xlm_tokenizer(
        target_text, return_tensors='pt', truncation=True, max_length=64
    ).to(device)
    with torch.no_grad():
        out = xlm_model(**enc)
    return out.last_hidden_state.squeeze(0).cpu().numpy().mean(0).astype(np.float32)


def find_target_span_xlm(target_raw, lang, sentence_text):
    """
    Use XLM-RoBERTa cosine similarity to locate the target word in sentence_text.

    Returns:
        best_start : word index of the best matching span start (or None)
        best_sim   : cosine similarity of the best span
        n_target   : number of words in the target phrase

    This is identical to the matching strategy in notebooks 02/03, preserving
    the validated 0.70 threshold for XLM-RoBERTa representations.
    """
    n_target     = len(target_raw.split())
    iso_vec      = get_xlm_isolated_vec(target_raw, lang)
    sent_vectors = get_xlm_word_vectors(sentence_text)  # list of (text, vec)

    if not sent_vectors:
        return None, 0.0, n_target

    vecs_only = [v for _, v in sent_vectors]
    best_sim, best_start = -2.0, 0
    for spos in range(max(1, len(vecs_only) - n_target + 1)):
        span_vec = np.mean(vecs_only[spos: spos + n_target], axis=0)
        sim      = cosine_sim(iso_vec, span_vec)
        if sim > best_sim:
            best_sim, best_start = sim, spos

    return best_start, best_sim, n_target


# ── GemmaX2 helpers (final embedding extraction) ─────────────────────────────────

def build_prefix_truncated(full_text, char_end):
    """
    Slice full_text[:char_end] and left-truncate at token boundary if needed.

    Returns:
        prefix_text : string to feed to GemmaX2
        truncated   : True if left-truncation was applied
        n_tokens    : number of tokens in prefix_text
    """
    prefix = full_text[:char_end]
    tokens = gemmax2_tokenizer.encode(prefix, add_special_tokens=True)

    if len(tokens) <= EFFECTIVE_CONTEXT_TOKENS:
        return prefix, False, len(tokens)

    # Keep rightmost tokens within budget, decode back to text
    kept        = tokens[-EFFECTIVE_CONTEXT_TOKENS:]
    prefix_text = gemmax2_tokenizer.decode(kept, skip_special_tokens=True)
    return prefix_text, True, len(kept)


def extract_gemmax2_embedding(prefix_text, target_components, lang, target_pos_hint=-1):
    """
    Run GemmaX2 on prefix_text and extract the hidden state for the target word.

    target_pos_hint: expected word position of target in prefix_text
    (-1 means last word, which is the default since the target is always
    appended at the end of the prefix).

    Uses character-span alignment since GemmaX2 tokenizer has no word_ids().

    Returns:
        vec         : (HIDDEN_DIM,) numpy array
        exact_match : True if target confirmed by string matching
    """
    phrase_len = len(target_components)

    enc = gemmax2_tokenizer(
        prefix_text,
        return_tensors='pt',
        return_offsets_mapping=True,
        truncation=True,
        max_length=MAX_TOKENS
    )
    offset_mapping = enc.pop('offset_mapping')[0].tolist()
    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        outputs = gemmax2_model(**enc, output_hidden_states=True)

    hidden = outputs.hidden_states[-1].squeeze(0).cpu().numpy()

    # Character-span alignment: map tokens back to whitespace-split words
    prefix_words    = prefix_text.split()
    word_char_spans = []
    pos = 0
    for word in prefix_words:
        try:
            start = prefix_text.index(word, pos)
        except ValueError:
            start = pos
        end = start + len(word)
        word_char_spans.append((start, end))
        pos = end

    word_token_vecs = [[] for _ in prefix_words]
    for tok_idx, (ts, te) in enumerate(offset_mapping):
        if ts == te:
            continue
        for w_idx, (ws, we) in enumerate(word_char_spans):
            if ts >= ws and te <= we + 1:
                word_token_vecs[w_idx].append(hidden[tok_idx])
                break

    window_token_list = []
    for w_idx, word in enumerate(prefix_words):
        vecs = word_token_vecs[w_idx]
        if vecs:
            avg_vec   = np.mean(vecs, axis=0).astype(np.float32)
            norm_text = normalize_text(word, lang)
            if norm_text:
                window_token_list.append((norm_text, avg_vec))

    # Search near expected target position (last word by default)
    if target_pos_hint < 0:
        target_pos = len(window_token_list) - 1
    else:
        target_pos = min(target_pos_hint, len(window_token_list) - 1)

    matched_at   = None
    search_order = list(range(
        max(0, target_pos - 3),
        min(len(window_token_list) - phrase_len + 1, target_pos + 3)
    ))
    search_order += [i for i in range(len(window_token_list) - phrase_len + 1)
                     if i not in search_order]

    for sp in search_order:
        if sp + phrase_len > len(window_token_list):
            continue
        if all(window_token_list[sp + j][0] == target_components[j]
               for j in range(phrase_len)):
            matched_at = sp
            break

    if matched_at is not None:
        vecs = [window_token_list[matched_at + j][1] for j in range(phrase_len)]
        return np.mean(vecs, axis=0), True
    else:
        # Positional fallback: target is last word in prefix by construction
        if window_token_list:
            return window_token_list[-1][1], False
        return np.zeros(HIDDEN_DIM, dtype=np.float32), False


print('Helper functions defined.')

## 7. Generate Embeddings for All Three Languages

In [ ]:
results = {}

for lang in LANGUAGES:
    print(f'\n{"="*60}')
    print(f'Processing {lang.upper()}...')
    print(f'{"="*60}')

    # Full context text and char-end lookup for this language
    if lang == 'en':
        full_text        = en_full_text
        lang_char_ends   = en_word_char_ends
        lang_sent_starts = None
        lang_sentences   = None
        print(f'  Context : podcast_transcript.csv ({len(full_df)} words)')
    elif lang == 'he':
        full_text        = he_full_text
        lang_sent_starts = he_sent_char_starts
        lang_sentences   = he_sentences
        print(f'  Context : 402 HE sentences concatenated')
    else:
        full_text        = ar_full_text
        lang_sent_starts = ar_sent_char_starts
        lang_sentences   = ar_sentences
        print(f'  Context : 402 AR sentences concatenated')

    # Pre-compute normalized target components
    split_char = '_' if lang == 'en' else ' '
    target_components_list = []
    for raw in content_df[lang].tolist():
        parts      = str(raw).strip().split(split_char)
        components = [normalize_text(p, lang) for p in parts if normalize_text(p, lang)]
        target_components_list.append(components)

    embeddings    = []
    exact_matches = 0
    fallbacks     = 0
    n_truncated   = 0
    n_dropped     = 0
    report_rows   = []

    for wi in tqdm(range(len(content_df)), desc=f'{lang.upper()}'):
        row        = content_df.iloc[wi]
        target_raw = str(row[lang]).strip()
        components = target_components_list[wi]

        sim       = 1.0    # default for EN (no cosine check needed)
        dropped   = False
        sent_idx  = None

        # ── Determine char_end: where in full_text does this word end? ─────────
        if lang == 'en':
            # Locate by timestamp in full transcript
            full_idx = find_full_transcript_idx(row['start'])
            char_end = lang_char_ends[full_idx]

        else:
            # HE/AR: assign sentence by timestamp, locate word via XLM-RoBERTa
            sent_idx = word_to_sent[wi]

            if sent_idx is None:
                dropped = True
            else:
                sent_text   = lang_sentences[sent_idx]
                best_start, sim, n_target = find_target_span_xlm(
                    target_raw, lang, sent_text
                )

                if sim < SIM_THRESHOLD:
                    dropped = True
                else:
                    # Find char end of matched span within sent_text
                    sent_words = sent_text.split()
                    pos2 = 0
                    char_in_sent = 0
                    for swi, sw in enumerate(sent_words):
                        try:
                            pos2 = sent_text.index(sw, pos2)
                        except ValueError:
                            pass
                        if swi == best_start + n_target - 1:
                            char_in_sent = pos2 + len(sw)
                            break
                        pos2 += len(sw)

                    # Global char end in the full concatenated HE/AR text
                    char_end = lang_sent_starts[sent_idx] + char_in_sent

        if dropped:
            embeddings.append(np.zeros(HIDDEN_DIM, dtype=np.float32))
            n_dropped += 1
            report_rows.append({
                'word_idx': wi, 'lang': lang, 'en': row['en'],
                'target': target_raw, 'sentence_idx': sent_idx,
                'n_context_toks': 0, 'truncated': False,
                'similarity': round(float(sim), 4),
                'exact_match': False, 'dropped': True
            })
            continue

        # ── Build GemmaX2 prefix (full context up to char_end) ───────────────────
        prefix_text, truncated, n_toks = build_prefix_truncated(full_text, char_end)
        if truncated:
            n_truncated += 1

        # ── Extract GemmaX2 embedding ─────────────────────────────────────────────
        vec, exact = extract_gemmax2_embedding(
            prefix_text, components, lang, target_pos_hint=-1
        )

        embeddings.append(vec)
        if exact:
            exact_matches += 1
        else:
            fallbacks += 1

        report_rows.append({
            'word_idx'      : wi,
            'lang'          : lang,
            'en'            : row['en'],
            'target'        : target_raw,
            'sentence_idx'  : sent_idx,
            'n_context_toks': n_toks,
            'truncated'     : truncated,
            'similarity'    : round(float(sim), 4),
            'exact_match'   : exact,
            'dropped'       : False,
        })

    results[lang] = {
        'embeddings'   : embeddings,
        'exact_matches': exact_matches,
        'fallbacks'    : fallbacks,
        'n_truncated'  : n_truncated,
        'n_dropped'    : n_dropped,
        'report_rows'  : report_rows,
    }

    n = len(content_df)
    print(f'\n  Done {lang.upper()}:')
    print(f'  Exact matches      : {exact_matches} / {n} ({exact_matches/n*100:.1f}%)')
    print(f'  Positional fallback: {fallbacks} / {n} ({fallbacks/n*100:.1f}%)')
    print(f'  Context truncated  : {n_truncated} / {n} ({n_truncated/n*100:.1f}%)')
    if lang != 'en':
        print(f'  Dropped (sim<{SIM_THRESHOLD})  : {n_dropped} / {n} ({n_dropped/n*100:.1f}%)')

print('\nAll languages processed.')

## 8. Report

In [ ]:
print('=' * 60)
print('GemmaX2-28-2B DUAL-MODEL EMBEDDING REPORT')
print('=' * 60)
print(f'GemmaX2 model     : {GemmaX2_MODEL_NAME}')
print(f'Locator model  : {XLM_MODEL_NAME}  (HE/AR word location only)')
print(f'Max tokens     : {MAX_TOKENS}')
print(f'Sim threshold  : {SIM_THRESHOLD}')
print(f'Total words    : {len(content_df)}')
print()

for lang in LANGUAGES:
    r    = results[lang]
    embs = np.array(r['embeddings'])
    rdf  = pd.DataFrame(r['report_rows'])
    kept = rdf[~rdf.get('dropped', pd.Series([False]*len(rdf)))]

    print(f'{lang.upper()}:')
    print(f'  Shape              : {embs.shape}')
    print(f'  Exact matches      : {r["exact_matches"]} ({r["exact_matches"]/len(content_df)*100:.1f}%)')
    print(f'  Positional fallback: {r["fallbacks"]} ({r["fallbacks"]/len(content_df)*100:.1f}%)')
    print(f'  Context truncated  : {r["n_truncated"]} ({r["n_truncated"]/len(content_df)*100:.1f}%)')
    if lang != 'en':
        print(f'  Dropped (sim<{SIM_THRESHOLD}) : {r["n_dropped"]} ({r["n_dropped"]/len(content_df)*100:.1f}%)')
        if len(kept):
            print(f'  Mean XLM sim score : {kept["similarity"].mean():.3f}')
    if len(kept):
        print(f'  Mean context toks  : {kept["n_context_toks"].mean():.1f}')
    print(f'  Any NaN            : {np.isnan(embs).any()}')
    assert embs.shape == (len(content_df), HIDDEN_DIM), f'Shape error {lang}'
    print(f'  Verified ✓')
    print()

print('Context source summary:')
print('  EN : podcast_transcript.csv — full 5136-word spoken stimulus')
print('  HE : 402 translated sentences concatenated — full discourse context')
print('  AR : 402 translated sentences concatenated — full discourse context')
print()
print('Word location summary:')
print('  EN : timestamp matching against full transcript (no cosine needed)')
print(f'  HE : XLM-RoBERTa cosine similarity (threshold {SIM_THRESHOLD})')
print(f'  AR : XLM-RoBERTa cosine similarity (threshold {SIM_THRESHOLD})')

## 9. Save

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

for lang in LANGUAGES:
    embs     = np.array(results[lang]['embeddings'])
    out_path = OUTPUT_PATHS[lang]
    pd.DataFrame(embs).to_csv(out_path, index=False)
    print(f'Saved {lang.upper()} -> {out_path.resolve()}  shape: {embs.shape}')

all_rows = []
for lang in LANGUAGES:
    all_rows.extend(results[lang]['report_rows'])
report_df = pd.DataFrame(all_rows)
report_df.to_csv(REPORT_PATH, index=False)
print(f'Saved report -> {REPORT_PATH.resolve()}  ({len(report_df)} rows)')

print()
print('Next steps:')
print('  1. Run 05_Projection_Residuals_Contextual.ipynb  (EMBEDDING_MODE = gemmax2)')
print('  2. Run 06_Encoding_Contextual.ipynb              (EMBEDDING_MODE = gemmax2)')
print('  3. Compare against notebook 04 (XLM-RoBERTa sliding window)')